In [32]:
# Vacant Land
import geopandas as gpd

vacant = gpd.read_file("/Users/sgaheer/Desktop/District 8 Data/Vacant_Land_Inventory.shp") # read in vacant shapefile 
dist = gpd.read_file("/Users/sgaheer/Desktop/District 8 Data/Council_District.shp") # read in district shapefile
dist8 = dist[dist["DISTRICT"] == "8"] # select district 8

# make sure they are in the same crs
print(dist8.crs)
print(vacant.crs)

EPSG:2227
EPSG:2227


In [33]:
# get only the vacant land within district 8
vacant_d8 = vacant[vacant.geometry.within(dist8.geometry.iloc[0])]

# print columns
print(vacant_d8.columns)

# write into a new shapefile
vacant_d8.to_file("/Users/sgaheer/Desktop/District 8 Data/vacant_d8.shp")


Index(['OBJECTID', 'FACILITYID', 'INTID', 'APN', 'GPDESIGNAT', 'VLIAREA',
       'LANDCLASS', 'PLANNINGAR', 'LASTUPDATE', 'NOTES', 'SHAPE_Leng',
       'SHAPE_Area', 'geometry'],
      dtype='object')


/Users/sgaheer/.pyenv/versions/3.12.1/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field LASTUPDATE create as date field, though DateTime requested.
  ogr_write(


In [41]:
# then, combine with manually collected data
import pandas as pd
import geopandas as gpd

data_2 =  pd.read_excel("/Users/sgaheer/Desktop/District 8 Data/Vacant_Plots_Updated_09_21.xlsx")

# check if APN is a string:
vacant_d8['APN'] = vacant_d8['APN'].astype(str)
data_2['APN'] = data_2['APN'].astype(str)

# join by APN

vacant_d8_final = pd.merge(vacant_d8, data_2, on='APN')

vacant_d8_final.columns = vacant_d8_final.columns.map(str)

# write into a new shapefile
vacant_d8_final.to_file("/Users/sgaheer/Desktop/District 8 Data/vacant_d8_final.shp")


/Users/sgaheer/.pyenv/versions/3.12.1/lib/python3.12/site-packages/geopandas/geodataframe.py:1981: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/var/folders/vq/97bmy6hd7pzfwr36kc4nlqbh0000gn/T/ipykernel_73399/2538150462.py:18: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  vacant_d8_final.to_file("/Users/sgaheer/Desktop/District 8 Data/vacant_d8_final.shp")
/Users/sgaheer/.pyenv/versions/3.12.1/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field LASTUPDATE create as date field, though DateTime requested.
  ogr_write(
/Users/sgaheer/.pyenv/versions/3.12.1/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered fi

In [ ]:
# double check total acreage:

import geopandas as gpd

data = gpd.read_file("/Users/sgaheer/Desktop/District 8 Data/vacant_d8_final.shp")

print(data["VLIAREA"].dtype)

# remove n/a
data = data[data["VLIAREA"] != "N/A"]

print(data["VLIAREA"].sum())



float64
296.31
Duplicate APNs: 0


In [23]:
# save as excel file

import geopandas as gpd

vacant_d8_final = vacant_d8_final.drop(columns='geometry')

vacant_d8_final.to_excel("/Users/sgaheer/Desktop/District 8 Data/vacant_d8_final.xlsx")


In [25]:
# print all unique stuff in zoning column:

print(vacant_d8_final["Zoning"].unique())

# sum all the "Planned Development" VLIAREA

print(vacant_d8_final[vacant_d8_final["Zoning"] == "Planned Development"]["VLIAREA"].sum())

['Planned Development' 'Agriculture'
 'Single-Family Residential (Up to Five Dwelling Units per Acre)'
 'Planned Development\n' 'Public/Quasi-Public (PQP)'
 'Planned Development (R-1-8 Low to Medium Density Residential Based District)'
 'Industrial Park'
 'Single-Family Residential (Up to Eight Dwelling Units per Acre) - Zoning Abbreviation R-1-8'
 nan 'Agriculture Commercial Pedestrian' 'Commercial Office'
 'Commercial Pedestrian'
 'Single-Family Residential (Up to Two Dwelling Units per Acre)'
 'Planned Development (R-1-1 Low to Medium Density Residential Based District)'
 'Rural Residential Residence (Up to One Dwelling Unit per Acre)'
 'Single-Family Residential (Up to Eight Dwelling Units per Acre)']
206.72


In [12]:
# Look at all Land Designations dataset

import geopandas as gpd

land_designations = gpd.read_file("/Users/sgaheer/Desktop/District 8 Data/General_Plan_2040/General_Plan_2040.shp")
parks = gpd.read_file("/Users/sgaheer/Desktop/District 8 Data/Park/Park.shp")
dist = gpd.read_file("/Users/sgaheer/Desktop/District 8 Data/Council_District.shp") # read in district shapefile
dist8 = dist[dist["DISTRICT"] == "8"] # select district 8

# Subset to District 8

land_designations_d8 = land_designations[land_designations.geometry.within(dist8.geometry.iloc[0])]
parks_d8 = parks[parks.geometry.within(dist8.geometry.iloc[0])]

print(land_designations_d8["GPDESIGNAT"].unique())

# THEN, subset to "Open Space, Parklands, Habitat."

land_designations_d8["GPDESIGNAT"] = land_designations_d8["GPDESIGNAT"].str.strip()
open_spaces_parks_d8 = land_designations_d8[land_designations_d8["GPDESIGNAT"] == "Open Space, Parklands and Habitat"]

# Cross reference the two dataframes and delete all land plots that are in the Parks dataset. We will end up with Open Spaces only. 

overlaps = gpd.sjoin(open_spaces_parks_d8, parks_d8, how="inner", predicate="intersects")

overlapping_ids = overlaps.index.unique()

#print(overlapping_ids)

# Remove overlapping open space areas to get Open Spaces only (excluding parks)
open_space_only = open_spaces_parks_d8[~open_spaces_parks_d8.index.isin(overlapping_ids)]

open_space_only.to_file("/Users/sgaheer/Desktop/District 8 Data/open_space_only_d8.shp")



['Public/Quasi-Public' 'Combined Industrial/Commercial'
 'Neighborhood/Community Commercial' 'Open Space, Parklands and Habitat'
 'Rural Residential' 'Lower Hillside' 'Open Hillside'
 'Private Recreation and Open Space' 'Residential Neighborhood'
 'Mixed Use Neighborhood' 'Industrial Park' 'Light Industrial'
 'Mobile Home Park' 'Regional Commercial']


/Users/sgaheer/.pyenv/versions/3.12.1/lib/python3.12/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
/Users/sgaheer/.pyenv/versions/3.12.1/lib/python3.12/site-packages/geopandas/geodataframe.py:1981: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/Users/sgaheer/.pyenv/versions/3.12.1/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field LASTUPDATE create as date field, though DateTime requested.
  ogr_write(


In [31]:
# print columns of land 

print(land_designations.columns)

print(parks.columns)

Index(['OBJECTID', 'FACILITYID', 'INTID', 'GPDESIGNAT', 'GPABBREVIA',
       'GP2040AREA', 'LASTUPDATE', 'NOTES', 'SHAPE_Leng', 'SHAPE_Area',
       'geometry'],
      dtype='object')
Index(['OBJECTID', 'FACILITYID', 'INTID', 'GISOBJID', 'NAME', 'ADDRESS',
       'PARKTYPE', 'PARKCLASS', 'SUBCLASS', 'CURRENTSTA', 'DATEOPENED',
       'DEEDDATE', 'COUNCILDIS', 'PARKDISTRI', 'ALLACRES', 'DEVACRES',
       'UNDEVACRES', 'OSACRES', 'WATERACRES', 'SUPERVISOR', 'SUPERVIS_1',
       'PARKMANAGE', 'MANAGERPHO', 'OWNER', 'NAMEALIAS', 'LOCATIONID',
       'PLANNINGAR', 'SPPLANAREA', 'PANELNUMBE', 'HYPERLINK', 'ADOPTABLE',
       'ADOPTIONST', 'PARKCOUNT', 'MAINTBY', 'CONTRACT', 'LASTUPDATE',
       'SHAPE_Leng', 'SHAPE_Area', 'geometry'],
      dtype='object')
